# Efficient Code Generation — Live Demo

**Columbia HPML Spring 2026** | Jasmine Truong · Yingxin Zhang · Arnav Mahajan · Jianyi Gao

**Two project cores — both live:**
1. **Model Quality** — Runtime-Aware SFT generates correct, efficient code
2. **System Efficiency** — GPU profiling reveals the bottleneck; vLLM is the fix

Run on **Colab A100**. Cells 1–2 are setup; run off-camera.

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes
!pip install --upgrade torchao>=0.16.0

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re, timeit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
RASFT_CKPT = "/content/drive/MyDrive/HPML Final Project/checkpoints_rerun/runtime_aware_full"

SYSTEM_PROMPT = (
    "Write a correct Python solution optimized for fast execution time. "
    "Use efficient algorithms and data structures to minimize runtime. "
    "Return only the code with no explanation."
)

# Two Sum: naive O(n²) nested loops vs efficient O(n) hash map
INSTRUCTION = (
    'def two_sum(nums, target):\n'
    '    """Given a list of integers and a target, return the indices of the\n'
    '    two numbers that add up to the target. Exactly one solution exists.\n'
    '\n'
    '    Example: two_sum([2, 7, 11, 15], 9) -> [0, 1]\n'
    '    """'
)

TESTS = """\
assert sorted(two_sum([2, 7, 11, 15], 9)) == [0, 1]
assert sorted(two_sum([3, 2, 4], 6)) == [1, 2]
assert sorted(two_sum([3, 3], 6)) == [0, 1]
assert sorted(two_sum([1, 2, 3, 4, 5], 9)) == [3, 4]
assert sorted(two_sum([0, 4, 3, 0], 0)) == [0, 3]"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": INSTRUCTION},
]

print("Setup complete.")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("Prompt ready.")

---
## Part 1 — Model Quality: Correct & Efficient Code

### The Problem: Two Sum

```python
def two_sum(nums, target):
    """Return indices of two numbers that add up to target."""
```

| Approach | Complexity |
|----------|-----------|
| Naive — nested loops | O(n²) |
| **Efficient — hash map** | **O(n)** |

**5 test cases.** Our Runtime-Aware SFT is prompted with an efficiency instruction — does it generate the fast solution?

In [ ]:
def clean_output(text, instruction):
    match = re.search(r'```(?:python)?\n(.*?)```', text, re.DOTALL)
    code = match.group(1).strip() if match else text.strip()
    if not re.match(r'^\s*def\s+', code):
        sig = instruction.strip().split('\n')[0]
        code = sig + '\n' + '\n'.join('    ' + l for l in code.splitlines())
    return code

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", trust_remote_code=True, torch_dtype=torch.float16
)
rasft = PeftModel.from_pretrained(base_model, RASFT_CKPT)
rasft.eval()
inputs = tokenizer(prompt, return_tensors="pt").to(rasft.device)

print("=" * 56)
print("  Runtime-Aware SFT — generating...")
print("=" * 56)
with torch.no_grad():
    out = rasft.generate(
        **inputs, max_new_tokens=256, do_sample=False,
        streamer=TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True),
    )

rasft_code = clean_output(
    tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True),
    INSTRUCTION,
)

print("\n--- Running 5 tests ---")
ns = {}
try:
    exec(rasft_code, ns)
    exec(TESTS, ns)
    print("✓  All 5 tests passed")
    t = timeit.timeit(lambda: ns["two_sum"]([2, 7, 11, 15], 9), number=100_000)
    print(f"⏱  Avg over 100k calls: {t / 100_000 * 1e6:.2f} µs per call")
except AssertionError:
    print("✗  Test failed — wrong indices returned")
except Exception as e:
    print(f"✗  Error: {type(e).__name__}: {e}")

In [ ]:
# Show what the model actually wrote — is it O(n) or O(n²)?
print("Generated code:")
print("─" * 56)
print(rasft_code)
print("─" * 56)
uses_hashmap = any(kw in rasft_code for kw in ("dict(", "{}", "seen", "lookup", "complement"))
print("✓  Uses O(n) hash-map approach" if uses_hashmap else "○  See code above for algorithm")

---
## Part 2 — System Efficiency: GPU Profiling → vLLM

The model generates correct code. But **how fast is serving it?**

PyTorch Profiler on an A100 reveals HuggingFace Transformers' hidden bottleneck — and vLLM eliminates it.

In [ ]:
# Profiling results from PyTorch Profiler on Vast.ai A100
print("━" * 60)
print("  GPU Profiling — HuggingFace generate() Bottlenecks")
print("━" * 60)
print("  CPU-bound decode loop   │  51% of step time on CPU")
print("  Per-token CPU-GPU syncs │  6.7% lost to aten::item (EOS check)")
print("  Tensor Core utilization │  only 6.7% during decode")
print("  GPU utilization (HF)    │  37.6%")
print()
print("  Root cause: Python's generate() runs one token at a time,")
print("  stalling the GPU between every forward pass.")
print()
print("━" * 60)
print("  FIX: Replace HuggingFace with vLLM")
print("  PagedAttention + continuous batching + CUDA graphs")
print("━" * 60)
print()
W = 28
print(f"  {'Metric':<{W}} {'HuggingFace':>12} {'vLLM':>10}")
print(f"  {'─'*W} {'─'*12} {'─'*10}")
print(f"  {'GPU Utilization':<{W}} {'37.6%':>12} {'97.2%':>10}  (+60 pp)")
print(f"  {'Throughput (tokens/s)':<{W}} {'234':>12} {'1782':>10}  (7.6×)")
print(f"  {'Per-prompt Latency':<{W}} {'0.69 s':>12} {'0.03 s':>10}  (21×)")
print(f"  {'GPU Memory Used':<{W}} {'3.6 GB':>12} {'36.7 GB':>10}")
print()
print("  Larger memory footprint = more sequences processed in parallel")
print("  = dramatically higher Tensor Core utilization.")
print("━" * 60)